In [1]:
import pandas as pd

df = pd.read_csv("src\data\df_final.csv")



<>:3: SyntaxWarning: invalid escape sequence '\d'
<>:3: SyntaxWarning: invalid escape sequence '\d'
C:\Users\diego\AppData\Local\Temp\ipykernel_24816\2969422670.py:3: SyntaxWarning: invalid escape sequence '\d'
  df = pd.read_csv("src\data\df_final.csv")


In [2]:
df.info()
df.head()
df.describe(include='all')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4562 entries, 0 to 4561
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   year        4562 non-null   float64
 1   estratopri  4540 non-null   object 
 2   aoj21       2908 non-null   object 
 3   colseg      2995 non-null   object 
 4   ur          1568 non-null   object 
 5   q1          1569 non-null   object 
 6   q2          4549 non-null   float64
 7   m1          2980 non-null   object 
 8   trust       1549 non-null   float64
 9   pr5         1476 non-null   float64
 10  ideologia   2347 non-null   object 
dtypes: float64(4), object(7)
memory usage: 392.2+ KB


,year,estratopri,aoj21,colseg,ur,q1,q2,m1,trust,pr5,ideologia
count,4562.000000,4540,2908,2995,1568,1569,4549.000000,2980,1549.000000,1476.000000,2347
unique,NaN,6,7,3,2,2,NaN,3,NaN,NaN,5
top,NaN,Oriental,Delincuentes comunes,Peor,Urbano,Mujer,NaN,Negativa,NaN,NaN,Centro
freq,NaN,1134,1298,2101,1428,796,NaN,1245,NaN,NaN,658
mean,2022.375712,NaN,NaN,NaN,NaN,NaN,40.792922,NaN,4.063267,3.939024,NaN
std,1.900278,NaN,NaN,NaN,NaN,NaN,14.233594,NaN,1.822165,1.827198,NaN
min,2021.000000,NaN,NaN,NaN,NaN,NaN,18.000000,NaN,1.000000,1.000000,NaN
25%,2021.000000,NaN,NaN,NaN,NaN,NaN,30.000000,NaN,3.000000,3.000000,NaN
50%,2021.000000,NaN,NaN,NaN,NaN,NaN,39.000000,NaN,4.000000,4.000000,NaN
75%,2025.000000,NaN,NaN,NaN,NaN,NaN,50.000000,NaN,5.000000,5.000000,NaN


In [3]:
df['colseg'].value_counts(dropna=False)
df['colseg'].value_counts(normalize=True)

colseg
Peor     0.701503
Igual    0.245409
Mejor    0.053088
Name: proportion, dtype: float64

In [4]:
for col in ['estratopri','aoj21','ur','q1','ideologia','m1','trust','pr5']:
    print(col, df[col].unique()[:20], '\n')

estratopri ['Oriental' 'Atlántica' 'Central' 'Bogotá' 'Amazonia' 'Pacífica' nan] 

aoj21 [nan 'Delincuentes comunes' 'Crimen organizado' 'Ninguno' 'Guerrillas'
 'Otros' 'Fuerzas de seguridad' 'Personas cercanas'] 

ur [nan 'Urbano' 'Rural'] 

q1 [nan 'Mujer' 'Hombre'] 

ideologia [nan 'Izquierda' 'Centro derecha' 'Derecha' 'Centro izquierda' 'Centro'] 

m1 [nan 'Negativa' 'Neutra' 'Positiva'] 

trust [nan  4.  7.  6.  3.  5.  2.  1.] 

pr5 [nan  3.  6.  5.  4.  2.  1.  7.] 



In [5]:
df.isna().sum()

year             0
estratopri      22
aoj21         1654
colseg        1567
ur            2994
q1            2993
q2              13
m1            1582
trust         3013
pr5           3086
ideologia     2215
dtype: int64

In [6]:
df['q2'].describe()

count    4549.000000
mean       40.792922
std        14.233594
min        18.000000
25%        30.000000
50%        39.000000
75%        50.000000
max        93.000000
Name: q2, dtype: float64

In [7]:
df.shape

(4562, 11)

In [8]:
df_model = df.copy()

# 1. Filtrar variable objetivo
df_model = df_model[df_model['colseg'].notna()]

# 2. Ordenar variable objetivo (CRÍTICO)
orden = ['Peor', 'Igual', 'Mejor']
df_model['colseg_ord'] = pd.Categorical(df_model['colseg'], categories=orden, ordered=True)

# 3. Convertir year a categórica
df_model['year'] = df_model['year'].astype(int).astype(str)

# 4. Edad limpia
df_model = df_model[df_model['q2'].notna()]

In [9]:
m1_order = ['Muy malo (pésimo)', 'Malo', 'Ni bueno, ni malo (regular)', 'Bueno', 'Muy bueno']
df_model['m1_ord'] = pd.Categorical(df_model['m1'], categories=m1_order, ordered=True).codes

In [ ]:
df_model[['year','estratopri','aoj21','q1','ur','m1','ideologia']].isna().mean()

year          0.000000
estratopri    0.002012
aoj21         0.035547
q1            0.482227
ur            0.482562
m1            0.013414
ideologia     0.221999
dtype: float64

In [22]:
import pandas as pd
import numpy as np
from statsmodels.miscmodels.ordinal_model import OrderedModel


df_model = df.copy()

#variables a usar
cols = ["colseg", "year", "estratopri", "q2", "m1"]
categoricas = ["estratopri", "m1"]
df_model = df_model[cols].copy()

#eliminar na (por siacaso)
df_model = df_model.dropna(subset=cols)

#ajustes tipo
df_model["year"] = df_model["year"].astype(int)
df_model["q2"] = pd.to_numeric(df_model["q2"], errors="coerce")


# codificacion colseg
orden_colseg = ["Peor", "Igual","Mejor"]

df_model = df_model[df_model["colseg"].isin(orden_colseg)].copy()

df_model["colseg_ord"] = pd.Categorical(
    df_model["colseg"],
    categories=orden_colseg,
    ordered=True
)

y = df_model["colseg_ord"].cat.codes # 0=Peor, 1=Igual, 2=Mejor

# dummies
X = pd.get_dummies(df_model[["year", "estratopri", "q2", "m1"]],
                   columns=categoricas,
                   drop_first=True)

X = X.astype(float)


model = OrderedModel(
    endog=y,
    exog=X,
    distr="logit"   
)

result = model.fit(method="bfgs", disp=True)


print(result.summary())

Optimization terminated successfully.
         Current function value: 0.669939
         Iterations: 88
         Function evaluations: 98
         Gradient evaluations: 98
                             OrderedModel Results                             
Dep. Variable:                      y   Log-Likelihood:                -1966.9
Model:                   OrderedModel   AIC:                             3956.
Method:            Maximum Likelihood   BIC:                             4022.
Date:                Mon, 13 Apr 2026                                         
Time:                        10:57:13                                         
No. Observations:                2936                                         
Df Residuals:                    2925                                         
Df Model:                           9                                         
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------